# Data audit

**What data do we actually have?**

Run this notebook top to bottom using the repository `.venv` kernel. Install the optional environment with
`uv sync --extra eda --extra dev`; no global kernel registration is needed when selecting `.venv/Scripts/python.exe`
in your notebook editor. Each notebook starts independently and discovers CSVs from `data/`.
Change `COHORTS` below to restrict the analysis. No reports, exports, or transformed data are written.
All visible outputs are aggregate; subject-level objects remain in memory.


In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.eda.io import load_tasks, cohort_tables

COHORTS = None  # Discover all years; use ["1year"] or ["2year", "3year"] to restrict.
tables = load_tasks(ROOT / "data", cohorts=COHORTS)
pd.set_option("display.max_rows", 24)
pd.set_option("display.max_columns", 12)
print("Loaded task files by cohort:", {c: len(g) for c, g in cohort_tables(tables).items()})


Loaded task files by cohort: {'1year': 21, '2year': 21, '3year': 18}


The raw audit preserves malformed tokens and unknown labels. Cohorts are separate populations; the same `Id` in two years does not establish the same child. Expected task counts come from the repository schema, not discovery limits.

In [2]:
from src.eda.audit import audit_tasks

audit = audit_tasks(tables)
display(audit.cohorts)


,cohort,task_files,expected_tasks,rows,subjects,subjects_in_every_task,identical_subject_sets,Gender_conflicting_subjects,Age_conflicting_subjects,Dominant_Hand_conflicting_subjects
0,1year,21,21,2935,140,136,False,0,0,0
1,2year,21,21,3045,145,145,True,0,0,0
2,3year,18,18,2610,145,145,True,0,0,0


In [3]:
inventory_columns = ["cohort", "task", "rows", "unique_subjects", "columns", "features", "all_zero_rows"]
display(audit.inventory[inventory_columns].set_index(["cohort", "task"]))
print("Every discovered CSV is represented:", len(audit.inventory) == len(tables))


rows  unique_subjects  columns  features  all_zero_rows
cohort task                                                            
1year  Task_1    140              140       36        30              1
       Task_2    140              140       36        30              1
       Task_3    140              140       36        30              1
       Task_4    140              140       36        30              1
       Task_5    140              140       36        30              1
...              ...              ...      ...       ...            ...
3year  Task_14   145              145       36        30             25
       Task_15   145              145       36        30             25
       Task_16   145              145       36        30             25
       Task_17   145              145       36        30             25
       Task_18   145              145       36        30             25

[60 rows x 5 columns]

Every discovered CSV is represented: True


Coverage and demographic agreement support within-cohort linkage but cannot independently verify acquisition IDs. Missing records and existing records with no finite or exclusively zero measurements are different conditions.

In [4]:
display(audit.coverage.loc[audit.coverage.missing_from_union.gt(0)])
display(audit.flags.groupby(["cohort", "issue"], sort=False)["count"].sum().unstack(fill_value=0).T)


,cohort,task,subjects,union_subjects,common_subjects,missing_from_union,matches_first_task
19,1year,Task_20,139,140,136,1,False
20,1year,Task_21,136,140,136,4,False


cohort,1year,2year,3year
issue,,,
all_zero_measurements,21,547,450
unknown_label_minus_one,568,547,487
implausible_or_missing_age,21,546,468
unrecognized_or_missing_category,42,1092,900
incomplete_subject_coverage,5,0,0


In [5]:
schema_issues = audit.schema.loc[
    ~audit.schema.present | audit.schema.invalid_numeric_cells.gt(0)
    | audit.schema.expected_dtype.eq("unrecognized_column")
]
display(schema_issues.head(20))
print("Schema issue entries:", len(schema_issues))
display(audit.flags.loc[audit.flags.issue.isin([
    "task_filename_mismatch", "multiple_task_values", "task_identifier_in_multiple_files",
    "missing_id", "duplicate_id_rows", "cross_task_demographic_conflict"
])].head(20))

display(audit.flags.loc[audit.flags.issue.isin(["implausible_or_missing_age", "unrecognized_or_missing_category"])]
        .groupby(["cohort", "issue", "detail"], sort=False)["count"].sum().to_frame("task_rows"))


,cohort,task,column,present,expected_dtype,invalid_numeric_cells


Schema issue entries: 0


,cohort,task,issue,count,detail


task_rows
cohort issue                            detail                      
1year  implausible_or_missing_age       Age='0.0'                 21
       unrecognized_or_missing_category Gender='0'                21
                                        Dominant_Hand='0'         21
2year  implausible_or_missing_age       Age='0.0'                546
       unrecognized_or_missing_category Gender='0'               546
                                        Dominant_Hand='0'        546
3year  implausible_or_missing_age       Age='0.0'                450
                                        Age='-4992.0'             18
       unrecognized_or_missing_category Gender='0'               450
                                        Dominant_Hand='0'        450

**Interpretation:** a zero-only record warrants manual review; it is retained. Ages outside the repository plausibility bounds and unrecognized demographic codes are flagged without recoding. Although older repository documentation interprets `-1` as absence, its meaning remains unresolved for this EDA. Confirm it with the data owner before defining eligibility. No final subject linkage or exclusion rule is established.